一、国家知识产权局（2015年及以后）专利相关数据的收集【专利申请代理状况、专利申请、授权按IPC分类分布状况、商标申请、注册及有效注册状况】

In [4]:
import requests
import pandas as pd
import time
import os

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36 Edg/148.0.0.0',
    'Referer': 'https://www.cnipa.gov.cn/',
    'Accept-Language': 'zh-CN,zh;q=0.9'
}

# 年份范围
YEARS = list(range(2015, 2025))

# 模块配置：模块代码 -> 模块名称
MODULES1 = {#2015-2018
    'd': '专利申请代理状况',
    'e': '专利申请、授权按IPC分类分布状况'
}
MODULES2 = {#2019-2021,2023-2024
    'd': '专利申请代理状况',
    'e': '专利申请、授权按IPC分类分布状况',
    'g': '商标申请、注册及有效注册状况'
}
MODULES3 = {#2022
    'd': '专利申请代理状况',
    'e': '专利申请、授权按IPC分类分布状况',
    'h': '商标申请、注册及有效注册状况'
}

#要提取的表格的下标
target_tables1 = [1,3]#2015-2019
target_tables2 = [1]#2020-2024

def get_html(url):
    """获取网页内容"""
    for attempt in range(3):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.encoding = 'utf-8'
            if response.status_code == 200:
                return response.text
            else:
                print(f"    HTTP {response.status_code}")
                return None
        except Exception as e:
            print(f"    重试 {attempt+1}: {str(e)[:50]}")
            time.sleep(2)
    return None

def extract_tables(html, target_indices=None):#提取指定下标的表格
    if not html:
        return []
    
    try:
        all_tables = pd.read_html(html)
        selected = []
        for idx in target_indices:
            selected.append(all_tables[idx])
            print(f"选中表格[{idx}]: {len(all_tables[idx])}行 x {all_tables[idx].shape[1]}列")
        return selected
        
    except Exception as e:
        print(f"    解析表格失败: {e}")
        return []

def merge_tables(tables):
    #合并表格
    if not tables:
        return None
    if len(tables) == 1:
        return tables[0]
    try:
        merged = pd.concat(tables, axis=0)
        print(f"按行合并：{merged.shape[0]}行 x {merged.shape[1]}列")
        return merged
    except Exception as e:
        print(f"表格合并失败：{e}, 将分别保存")
        return tables

def save_tables(tables, year, module_name, sub_page):
    """保存表格"""
    if tables is None:
        return 0
    #创建目录
    os.makedirs(f'ex1_1/{year}', exist_ok=True)
    saved = 0
    #若tables是DataFrame
    if isinstance(tables, pd.DataFrame):
        filename = f"ex1_1/{year}/{module_name}_{sub_page}_table.csv"
        tables.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"    保存文件: {filename} ({len(tables)}行)")
        saved = 1
    #若tables是list(包含多个表格)，分别保存
    elif isinstance(tables, list):
        for i, df in enumerate(tables):
            filename = f"ex1_1/{year}/{module_name}_{sub_page}_table{i+1}.csv"
            df.to_csv(filename, index=False, encoding='utf-8-sig')
            print(f"    保存: {filename} ({len(tables)}行)")
            saved += 1
    else:
        print(f"警告: 不支持的类型{type(tables)}, 跳过保存")
    return saved

def crawl_year(year):
    """爬取单个年份的所有模块数据"""
    print(f"\n{'='*60}")
    print(f"处理年份: {year}")
    print(f"{'='*60}")

    if year < 2020:
        target_tables = target_tables1
    else:
        target_tables = target_tables2
    print(f"提取表格下标={target_tables}, 按行堆叠")
    
    total_tables = 0
    if year < 2019:
        MODULES = MODULES1
    elif year == 2022:
        MODULES = MODULES3
    else:
        MODULES = MODULES2

    for module_code, module_name in MODULES.items():
        print(f"\n模块: {module_name} ({module_code})")
        
        # 尝试爬取子页面 d1, d2, d3... 直到失败
        sub_page = 1
        while True:
            url = f"https://www.cnipa.gov.cn/tjxx/jianbao/year{year}/{module_code}/{module_code}{sub_page}.html"
            print(f"  尝试: {url}")
            
            html = get_html(url)
            if not html:
                break
            
            tables = extract_tables(html, target_tables)
            if tables:
                merged = merge_tables(tables)
                saved = save_tables(merged, year, module_name, f"{module_code}{sub_page}")
                total_tables += saved
            else:
                print(f"    当前页面无表格")
            
            sub_page += 1
            time.sleep(0.5)
    
    return total_tables

def main():
    print("开始爬取国家知识产权局统计数据")
    print(f"目标年份: {YEARS}")

    total = 0
    for year in YEARS:
        tables_count = crawl_year(year)
        total += tables_count
        time.sleep(2)
    
    print(f"\n{'='*60}")
    print(f"爬取完成！共保存 {total} 个表格")
    print(f"文件保存在 ex1_1/ 目录")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

开始爬取国家知识产权局统计数据
目标年份: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

处理年份: 2015
提取表格下标=[1, 3], 按行堆叠

模块: 专利申请代理状况 (d)
  尝试: https://www.cnipa.gov.cn/tjxx/jianbao/year2015/d/d1.html


C:\Users\admin.DESKTOP-K6PI601\AppData\Local\Temp\ipykernel_18368\2928208772.py:56: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  all_tables = pd.read_html(html)


选中表格[1]: 1行 x 1列
选中表格[3]: 53行 x 8列
按行合并：54行 x 8列
    保存文件: ex1_1/2015/专利申请代理状况_d1_table.csv (54行)
  尝试: https://www.cnipa.gov.cn/tjxx/jianbao/year2015/d/d2.html


C:\Users\admin.DESKTOP-K6PI601\AppData\Local\Temp\ipykernel_18368\2928208772.py:56: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  all_tables = pd.read_html(html)


选中表格[1]: 1行 x 1列
选中表格[3]: 53行 x 8列
按行合并：54行 x 8列
    保存文件: ex1_1/2015/专利申请代理状况_d2_table.csv (54行)
  尝试: https://www.cnipa.gov.cn/tjxx/jianbao/year2015/d/d3.html


...

选中表格[1]: 56行 x 11列
    保存文件: ex1_1/2024/商标申请、注册及有效注册状况_g2_table.csv (56行)
  尝试: https://www.cnipa.gov.cn/tjxx/jianbao/year2024/g/g3.html


C:\Users\admin.DESKTOP-K6PI601\AppData\Local\Temp\ipykernel_18368\2928208772.py:56: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  all_tables = pd.read_html(html)


选中表格[1]: 210行 x 11列
    保存文件: ex1_1/2024/商标申请、注册及有效注册状况_g3_table.csv (210行)
  尝试: https://www.cnipa.gov.cn/tjxx/jianbao/year2024/g/g4.html


C:\Users\admin.DESKTOP-K6PI601\AppData\Local\Temp\ipykernel_18368\2928208772.py:56: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  all_tables = pd.read_html(html)


选中表格[1]: 56行 x 5列
    保存文件: ex1_1/2024/商标申请、注册及有效注册状况_g4_table.csv (56行)
  尝试: https://www.cnipa.gov.cn/tjxx/jianbao/year2024/g/g5.html
    HTTP 404

爬取完成！共保存 139 个表格
文件保存在 ex1_1/ 目录


二、佰腾网不同类型2000-2024年专利数据收集

In [ ]:
import requests
import pandas as pd
import time
import random

# ==================== 请填写你的Cookie ====================
COOKIE_STRING = """_c_WBKFRo=zPrkSTwOZLJv8kiU5A6lhPdFobdKACSjjYnQL4WC; JSESSIONID=7B5BCBC7FC99AFFB7A1699B6E5321897; Hm_lvt_7fc44f078bf7b5e19489428c362109a3=1778568393,1778685023; HMACCOUNT=8745B4C830D6AE9C; BSESSION=dbce038bad527a599f6cb9219e6f50ffa970a9df4490429e; token=e0b4de44337c2bca; Hm_lpvt_7fc44f078bf7b5e19489428c362109a3=1778685348; yunsuo_session_verify=2b823556dce5a6e65ba5ca56151534eb"""
# =========================================================

def get_data(region="江苏"):
    """获取单个地区的专利数据"""
    
    url = "https://api.db.baiten.cn/patdatabase/open/bt/pat/nav/store/node/pat/condition/filter"
    
    headers = {
        "accept": "application/json, text/plain, */*",
        "content-type": "application/json;charset=UTF-8",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    }
    
    payload = {
        "query": f"aa:(({region})) AND (pd:[20000101 TO 20241231])",
        "code": "cty_type",
        "sc": "35184372088831",
        "token": "d782f8d891caba24ff6aeba315180caczik7x5ndl9a9yzfey9eu8e620ec2980286f3de2d546df9be6a4332720d5be6294389",
        "key": "e0b4de44337c2bca",
        "userKey": "e0b4de44337c2bca"
    }
    
    def parse_cookie(c):
        return dict([x.strip().split('=', 1) for x in c.split(';') if '=' in x])
    
    time.sleep(random.uniform(2, 3))
    
    resp = requests.post(url, headers=headers, cookies=parse_cookie(COOKIE_STRING), json=payload)
    data = resp.json()
    
    if data.get("code") == 0:
        cn = data["data"]["facetPivot"]["cn"]
        return {
            "地区": region,
            "实用新型": cn["cn_um"],
            "发明公开": cn["cn_in_gp"],
            "发明授权": cn["cn_gp"],
            "发明合计": cn["cn_in"],
            "外观设计": cn["cn_id"],
            "总计": cn["cn_um"] + cn["cn_in"] + cn["cn_id"]
        }
    return None

# 查询多个地区
regions = ["江苏", "浙江", "上海", "四川", "北京"]

results = []
for region in regions:
    print(f"查询 {region}...")
    r = get_data(region)
    if r:
        results.append(r)

# 保存到Excel
df = pd.DataFrame(results)
df.to_excel("ex1_2专利统计.xlsx", index=False)
print(f"\n✅ 已保存到 ex1_2专利统计.xlsx")
print(df)

查询 江苏...
查询 浙江...
查询 上海...
查询 四川...
查询 北京...

✅ 已保存到 ex1_2专利统计.xlsx
   地区     实用新型     发明公开    发明授权     发明合计     外观设计       总计
0  江苏  3185951  1521919  825228  2347147  1286644  6819742
1  浙江  2406956   750777  583958  1334735  1494712  5236403
2  上海   944258   561582  416892   978474   313776  2236508
3  四川   724354   341387  232634   574021   319524  1617899
4  北京   884186   851292  927305  1778597   247241  2910024


三、2024-2025年4月中某月任一省份知产官方微博数据爬取，包括内容、评论、点赞数、转发数

In [1]:
import requests
import json
import re
import time
import pandas as pd
from datetime import datetime

# ==================== 请填入你的 Cookies ====================
COOKIES = {
    "SUB": "_2A25HB1O_DeRhGeVJ7lAY8izMwjiIHXVkfel3rDV8PUJbkNAYLVCnkW1NT_VQE6AQkXADVYmr3dleyhwe2EbykhET",
    "SUBP": "0033WrSXqPxfM725Ws9jqgMF55529P9D9WWQSYDLMfaRq.JhdqF6hTX25JpX5KzhUgL.FoeNSKz4eoz71KB2dJLoI7yBMJL09PxXwBtt",
}
# ============================================================

UID = "3234994367"
START_DATE = "2025-01-01"
END_DATE = "2025-01-31"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": f"https://weibo.com/u/{UID}",
    "x-requested-with": "XMLHttpRequest",
}


def parse_weibo_time(time_str):
    """解析微博时间"""
    if not time_str:
        return None
    try:
        parts = time_str.split(' +0800 ')
        if len(parts) == 2:
            time_str = parts[0] + ' ' + parts[1]
        return datetime.strptime(time_str, "%a %b %d %H:%M:%S %Y")
    except:
        return None


def get_weibo_list(page):
    """按页码获取微博"""
    url = "https://weibo.com/ajax/statuses/mymblog"
    params = {"uid": UID, "feature": 0, "page": page, "count": 20}
    try:
        r = requests.get(url, params=params, cookies=COOKIES, headers=headers, timeout=10)
        return r.json()
    except Exception as e:
        print(f"请求失败: {e}")
        return None


def get_comments(mid):
    """获取评论"""
    url = "https://weibo.com/ajax/statuses/buildComments"
    params = {"mid": mid, "max_id": 0, "count": 30}
    try:
        r = requests.get(url, params=params, cookies=COOKIES, headers=headers, timeout=10)
        data = r.json()
        comments = []
        for c in data.get("data", {}).get("comments", []):
            text = c.get("text", "")
            text = re.sub(r'<[^>]+>', '', text)
            comments.append({
                "user": c.get("user", {}).get("screen_name", ""),
                "text": text,
                "like": c.get("like_count", 0),
            })
        return comments
    except:
        return []


def main():
    start = datetime.strptime(START_DATE, "%Y-%m-%d")
    end = datetime.strptime(END_DATE, "%Y-%m-%d")
    
    print(f"开始爬取 {START_DATE} 至 {END_DATE}")
    print("=" * 60)
    
    all_weibos = []
    
    # 翻第1页到第10页
    for page in range(1, 11):
        print(f"\n正在获取第 {page} 页...")
        data = get_weibo_list(page)
        
        if not data or data.get("ok") != 1:
            print("请求失败或 Cookie 无效")
            break
        
        weibo_list = data.get("data", {}).get("list", [])
        if not weibo_list:
            print("没有更多数据了")
            break
        
        # 显示本页时间范围
        if weibo_list:
            first_time = weibo_list[0].get("created_at", "未知")
            last_time = weibo_list[-1].get("created_at", "未知")
            print(f"本页时间范围: {first_time} 至 {last_time}")
        
        for weibo in weibo_list:
            created_at = weibo.get("created_at", "")
            weibo_time = parse_weibo_time(created_at)
            
            if not weibo_time:
                continue
            
            # 筛选目标时间范围
            if start <= weibo_time <= end:
                text = weibo.get("text_raw", "") or weibo.get("text", "")
                text = re.sub(r'<[^>]+>', '', text).strip()
                
                weibo_data = {
                    "id": weibo.get("id"),
                    "time": created_at,
                    "text": text,
                    "like": weibo.get("like_count", 0),
                    "repost": weibo.get("reposts_count", 0),
                    "comment_count": weibo.get("comment_count", 0),
                    "comments": []
                }
                
                # 获取评论
                if weibo_data["comment_count"] > 0 and weibo_data["comment_count"] <= 50:
                    print(f"  获取评论中...")
                    weibo_data["comments"] = get_comments(str(weibo.get("id", "")))
                    print(f"  获取到 {len(weibo_data['comments'])} 条评论")
                
                all_weibos.append(weibo_data)
                print(f"  ✓ {created_at} | {text[:40]}... | 赞:{weibo_data['like']} 转:{weibo_data['repost']} 评:{weibo_data['comment_count']}")
        
        time.sleep(1)  # 避免请求过快
    
    # 保存为excel
    rows = []
    for w in all_weibos:
        rows.append({
            "发布时间": w["time"], 
            "微博内容": w["text"], 
            "点赞数": w["like"], 
            "转发数": w["repost"], 
            "评论数": w["comment_count"], 
            "评论内容": "\n".join([c["text"] for c in w["comments"]])
        })

    df = pd.DataFrame(rows)
    output_file = f"ex1_3上海知识产权_{START_DATE}_to_{END_DATE}.xlsx"
    df.to_excel(output_file, index=False, engine="openpyxl")
    
    print(f"\n{'='*60}")
    print(f"爬取完成！共 {len(all_weibos)} 条微博")
    print(f"已保存到: {output_file}")
    
    if len(all_weibos) == 0:
        print(f"\n⚠️ 没有找到 {START_DATE} 至 {END_DATE} 之间的微博")


if __name__ == "__main__":
    main()

开始爬取 2025-01-01 至 2025-01-31

正在获取第 1 页...
本页时间范围: Fri Jan 17 11:14:38 +0800 2020 至 Mon Jan 26 15:54:11 +0800 2026

正在获取第 2 页...
本页时间范围: Thu Jan 22 17:35:37 +0800 2026 至 Mon Oct 20 11:08:26 +0800 2025

正在获取第 3 页...
本页时间范围: Mon Oct 20 09:38:23 +0800 2025 至 Wed Apr 30 17:20:50 +0800 2025

正在获取第 4 页...
本页时间范围: Tue Apr 22 13:48:16 +0800 2025 至 Mon Nov 25 11:32:24 +0800 2024
  ✓ Tue Jan 07 16:34:53 +0800 2025 | 秒懂！2025年全国知识产权局局长会议工作报告 ​​​... | 赞:0 转:0 评:0
  ✓ Thu Jan 02 16:08:43 +0800 2025 | 87件注册商标纳入第十六批上海市重点商标保护名录 ​​​... | 赞:0 转:0 评:0

正在获取第 5 页...
本页时间范围: Fri Nov 15 09:10:25 +0800 2024 至 Fri Jun 14 10:32:46 +0800 2024

正在获取第 6 页...
本页时间范围: Wed May 29 15:18:48 +0800 2024 至 Tue Jan 09 17:20:29 +0800 2024

正在获取第 7 页...
本页时间范围: Fri Jan 05 17:14:39 +0800 2024 至 Wed Sep 27 14:10:02 +0800 2023

正在获取第 8 页...
本页时间范围: Fri Sep 15 14:26:16 +0800 2023 至 Tue Dec 20 22:43:11 +0800 2022

正在获取第 9 页...
本页时间范围: Mon Dec 19 15:51:24 +0800 2022 至 Mon Jul 04 22:25:44 +0800 2022

正在获取第 10 页...
本页时间范围: Tue Jul 0

四、河北新闻网一天数据

In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import re
import time

base_url = "https://xczx.hebnews.cn"
save_dir = "ex1_4"
os.makedirs(save_dir, exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

def safe_filename(text):
    """去除文件名中的非法字符"""
    return re.sub(r'[\\/*?:"<>|]', '', text).strip()

def get_soup(url):
    """获取网页的BeautifulSoup对象"""
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        resp.encoding = "utf-8"
        return BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        print(f"请求失败：{url}，错误：{e}")
        return None

def get_plates(soup):
    """根据HTML结构获取所有板块的链接和标题"""
    plates = []
    # 板块位于 .min_box_left 容器内
    left_box = soup.select_one(".min_box_left")
    if not left_box:
        print("未找到板块容器 .min_box_left")
        return plates
    
    # 找到所有 li 标签中的 a 链接
    for li in left_box.find_all("li"):
        a_tag = li.find("a")
        if not a_tag:
            continue
        href = a_tag.get("href")
        title = a_tag.get_text(strip=True)
        if href and title:
            # 补全链接
            if not href.startswith("http"):
                href = base_url + "/" + href.lstrip("/")
            plates.append({"title": title, "url": href})
    
    return plates

def get_articles_from_plate(plate_url):
    """从一个板块页面获取文章列表"""
    soup = get_soup(plate_url)
    if not soup:
        return []
    articles = []

    for feed in soup.select(".min_feed"):
        #提取标题和链接
        h2 = feed.find("h2")
        if not h2:
            continue
        a_tag = h2.find("a")
        if not a_tag:
            continue
        href = a_tag.get("href")
        title = a_tag.get_text(strip=True)
        #提取简介
        p_tag = feed.find("p")
        summary = p_tag.get_text(strip=True) if p_tag else ""
        #提取时间
        span_tag = feed.find("span")
        pub_time = span_tag.get_text(strip=True) if span_tag else ""
        #补全链接
        if href and not href.startswith("http"):
                href = base_url + "/" + href.lstrip("/")

        if href and title: 
            articles.append({
                "title": title,
                "url": href,
                "time": pub_time,
                "summary": summary
            })
    return articles[:20]  # 每个板块取前20篇

def get_article_detail(article_url):
    """抓取文章正文和作者"""
    try:
        resp = requests.get(article_url, headers=headers, timeout=10)
        resp.encoding = "utf-8"
        html = resp.text
        soup = BeautifulSoup(html, "html.parser")
        
        # 正文区域
        content_div = soup.select_one(".min_con_text, #p1")
        content = ""
        if content_div:
            for tag in content_div(["script", "style"]):
                tag.decompose()
            content = content_div.get_text("\n", strip=True)
        
        # 用正则从 enpproperty 注释中提取作者
        author = ""
        prop_match = re.search(r'<!--enpproperty(.*?)-->', html, re.DOTALL)
        if prop_match:
            prop_text = prop_match.group(1)
            author_match = re.search(r'<author>(.*?)</author>', prop_text)
            author = author_match.group(1) if author_match else ""
        return content, author
    except Exception as e:
        print(f"抓取文章失败：{article_url}，错误：{e}")
        return "", ""

def save_article(title, pub_date, author, summary, content, url):
    """保存文章到本地"""
    filename = f"{pub_date}_{title}" if pub_date else title
    filename = safe_filename(filename)[:100]  # 限制文件名长度
    if not filename:
        filename = f"article_{int(time.time())}"
    filepath = os.path.join(save_dir, f"{filename}.txt")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"标题：{title}\n")
        f.write(f"发布时间：{pub_date}\n")
        f.write(f"作者：{author}\n")
        f.write(f"简介：{summary}\n")
        f.write(f"原文链接：{url}\n")
        f.write("\n正文：\n")
        f.write(content)
    print(f"已保存：{filepath}")

def main():
    print("开始抓取河北新闻网...")
    soup = get_soup(base_url)
    if not soup:
        print("首页获取失败")
        return
    
    # 1. 获取所有板块
    plates = get_plates(soup)
    print(f"共找到 {len(plates)} 个板块")
    for p in plates:
        print(f"  - {p['title']}: {p['url']}")
    
    # 2. 遍历6个板块
    for plate in plates[:6]: 
        print(f"\n正在处理板块：{plate['title']}")
        articles = get_articles_from_plate(plate["url"])
        print(f"  找到 {len(articles)} 篇文章")
        
        for idx, art in enumerate(articles[:20], 1):  # 每个板块抓20篇
            print(f"    正在抓取第 {idx} 篇：{art['title'][:30]}...")
            content, author = get_article_detail(art["url"])
            if content:
                save_article(title=art['title'], pub_date=art['time'], author=author, 
                             summary=art['summary'], content=content, url=art['url'])
            else:
                print(f"    正文为空，跳过")
            time.sleep(0.5)  # 避免请求过快

if __name__ == "__main__":
    main()

开始抓取河北新闻网...
共找到 6 个板块
  - 产业兴旺: https://xczx.hebnews.cn/node_147686.htm
  - 休闲农业: https://xczx.hebnews.cn/node_147690.htm
  - 品牌强农: https://xczx.hebnews.cn/node_363925.htm
  - 和美乡村: https://xczx.hebnews.cn/node_363926.htm
  - 科技农业: https://xczx.hebnews.cn/node_364524.htm
  - 农安河北: https://xczx.hebnews.cn/node_364539.htm

正在处理板块：产业兴旺
  找到 20 篇文章
    正在抓取第 1 篇：石家庄市鹿泉区第六届谷家峪香椿节开幕...
已保存：ex1_4\2026-04-04 212205_石家庄市鹿泉区第六届谷家峪香椿节开幕.txt
    正在抓取第 2 篇：校企携手 赋能种业｜齐鲁师范学院助力南宫高质量发展合作对接会...
已保存：ex1_4\2026-04-03 180141_校企携手 赋能种业｜齐鲁师范学院助力南宫高质量发展合作对接会在南宫召开.txt
    正在抓取第 3 篇：石家庄达西创业孵化基地举办智能AI数据采集系统对接交流会...
已保存：ex1_4\2026-04-01 101822_石家庄达西创业孵化基地举办智能AI数据采集系统对接交流会.txt
    正在抓取第 4 篇：河北省农业企业联合会换届大会在石家庄召开...
已保存：ex1_4\2026-03-31 124106_河北省农业企业联合会换届大会在石家庄召开.txt
    正在抓取第 5 篇：“建台账强回访” 邢台市信都区行政审批局企业服务中心跑出服务...
已保存：ex1_4\2026-03-28 084720_“建台账强回访” 邢台市信都区行政审批局企业服务中心跑出服务“加速度”.txt
    正在抓取第 6 篇：协助构建产业全链数字化新生态，引领邱县食品产业高质量发展...
已保存：ex1_4\2026-02-14 152525_协助构建产业全链数字化新生态，引领邱县食品产业高质量发展.txt
    正在抓取第 7 篇：春潮涌动启新程 科技赋农谱新